In [2]:
import logging
from pathlib import Path
import sys

from modules import plotting
from modules import SequenceRepresentation as sr
from modules import training

2025-03-18 19:55:44.532113: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-18 19:55:44.650268: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-18 19:55:45.093904: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-18 19:55:46.362149: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", 
                    encoding='utf-8', level=logging.DEBUG)
#logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

In [4]:
#wd = Path("/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
wd = Path("/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
experiment_dirs = [f for f in wd.iterdir() if f.is_dir() and not (f.name.startswith("test") or f.name.startswith("slurm"))]
print(experiment_dirs[:max(3, len(experiment_dirs))])

[PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak'), PosixPath('/home/matthis/PhD/mnt/brain/g

In [5]:
def evaluate(experiment_dirs, evaluator_path, neg_evaluator_path = None):
    n_test_seqs = 0 # track total number of test seqs over all experiments to compare between runs, even if single experiments fail
    n_peaks = {}
    n_skipped_seqs = 0
    n_skipped_peaks = {}
    skipped_experiments = []
    glob_seqs = {}
    glob_seqs_neg = {}
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
        n_test_seqs += sum([len(g) for g in testdata])
        if skipping:
            n_skipped_seqs += sum([len(g) for g in testdata])
            skipped_experiments.append(str(ed))

        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in n_peaks:
                        n_peaks[peaksrc] = 0
                    n_peaks[peaksrc] += 1
                    if skipping:
                        if peaksrc not in n_skipped_peaks:
                            n_skipped_peaks[peaksrc] = 0
                        n_skipped_peaks[peaksrc] += 1

        if not (ed / evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / evaluator_path} does not exist")
            continue

        if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / neg_evaluator_path} does not exist")
            continue
        
        # retrieve peaks from test data (peaks are stored as genomic elements in the sequences)
        peaks = {}
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in peaks:
                        peaks[peaksrc] = {}
                    if s.id not in peaks[peaksrc]:
                        peaks[peaksrc][s.id] = []
                    peakstart, peakend = e.getRelativePositions(s)
                    assert peakend == peakstart + 1, f"Peak {e} is not a single base pair"
                    peaks[peaksrc][s.id].append(peakstart)

        # store how many times and where each sequence was hit
        seqdict = {s.id: [] for g in testdata for s in g} 
        evaluator = training.loadMultiTrainingEvaluation(str(ed / evaluator_path), testdata)
        assert len(evaluator.trainings) == 1
        tr = evaluator.trainings[0]
        for link in tr.links:
            for occs in link.occs: # list of list of occurrences
                for occ in occs:
                    assert occ.sequence.id in seqdict
                    seqdict[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        if neg_evaluator_path is not None:
            # store how many times and where each sequence was hit
            assert (ed / 'negative_test_sequences_0.json').exists()
            testdata_neg = sr.loadJSONGenomeList(str(ed / 'negative_test_sequences_0.json'))
            seqdict_neg = {s.id: [] for g in testdata_neg for s in g} 
            evaluator_neg = training.loadMultiTrainingEvaluation(str(ed / neg_evaluator_path), testdata_neg)
            assert len(evaluator_neg.trainings) == 1
            tr = evaluator_neg.trainings[0]
            for link in tr.links:
                for occs in link.occs: # list of list of occurrences
                    for occ in occs:
                        assert occ.sequence.id in seqdict_neg
                        seqdict_neg[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        # globally count how many times each sequence was hit
        for sid in seqdict:
            if sid not in glob_seqs:
                glob_seqs[sid] = {'hits': 0, 'peaks': {}}
            glob_seqs[sid]['hits'] += len(seqdict[sid])
            for peaksrc in peaks:
                if sid in peaks[peaksrc]:
                    if peaksrc not in glob_seqs[sid]['peaks']:
                        glob_seqs[sid]['peaks'][peaksrc] = {'peaks': set(), 'hits': set()}
                    glob_seqs[sid]['peaks'][peaksrc]['peaks'].update(peaks[peaksrc][sid])
                    for p in peaks[peaksrc][sid]:
                        for hit in seqdict[sid]:
                            if hit[0] <= p < hit[1]:
                                glob_seqs[sid]['peaks'][peaksrc]['hits'].add(p)

        if neg_evaluator_path is not None:
            for sid in seqdict_neg:
                if sid not in glob_seqs_neg:
                    glob_seqs_neg[sid] = {'hits': 0}
                glob_seqs_neg[sid]['hits'] += len(seqdict_neg[sid])

    nseqs = len(glob_seqs.keys())
    nseqs_hit = len([k for k in glob_seqs.keys() if glob_seqs[k]['hits'] > 0])
    nmatches = sum([v['hits'] for v in glob_seqs.values()])

    print(f"Total number of test sequences: {n_test_seqs} | Number of peaks in these sequences: {n_peaks}")
    print(f"Skipped number of test sequences: {n_skipped_seqs} | Number of peaks in these sequences: {n_skipped_peaks}")
    print(f"Skipped {len(skipped_experiments)}/{len(experiment_dirs)} experiments: {skipped_experiments}")
    print(f"Number of sequences: {nseqs}")
    print(f"Number of sequences with hits: {nseqs_hit} | ratio: {nseqs_hit/nseqs:.2f}")
    print(f"Number of matches: {nmatches} | ratio: {nmatches/nseqs:.2f}")
    print()

    for peaksrc in peaks:
        print(f"Peak source: {peaksrc}")
        n_peak_seqs = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks']])
        n_peak_seqs_with_hits = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks'] and glob_seqs[k]['peaks'][peaksrc]['hits']])
        n_peaks = sum([len(v) for v in peaks[peaksrc].values()])
        n_peaks_hit = sum([len(v['peaks'][peaksrc]['hits']) for v in glob_seqs.values() if peaksrc in v['peaks']])

        print(f"Number of sequences with peaks: {n_peak_seqs}")
        print(f"Number of sequences with hits on peaks: {n_peak_seqs_with_hits} | ratio: {n_peak_seqs_with_hits/n_peak_seqs:.2f}")
        print(f"Number of peaks: {n_peaks}")
        print(f"Number of hits on peaks: {n_peaks_hit} | ratio: {n_peaks_hit/n_peaks:.2f}")
        print()

    if neg_evaluator_path is not None:
        nseqs_neg = len(glob_seqs_neg.keys())
        nseqs_hit_neg = len([k for k in glob_seqs_neg.keys() if glob_seqs_neg[k]['hits'] > 0])
        nmatches_neg = sum([v['hits'] for v in glob_seqs_neg.values()])

        print(f"Number of negative sequences: {nseqs_neg}")
        print(f"Number of negative sequences with hits: {nseqs_hit_neg} | ratio: {nseqs_hit_neg/nseqs_neg:.2f}")
        print(f"Number of negative matches: {nmatches_neg} | ratio: {nmatches_neg/nseqs_neg:.2f}")
        print()

In [6]:
evaluate(experiment_dirs, 'evaluator_test.json', 'evaluator_negative_test.json')

2025-03-18 19:56:17,743 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-18 19:56:18,317 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)
2025-03-18 19:56:22,625 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:42295995-42296645:0:0-650 (2x), chr2:145089806-145090327:0:0-521 (2x), chr3:42641896-42642557:0:0-661 (2x)
2025-03-18 19:56:23,690 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:42295995-42296645:0:0-650 (2x), negative_chr2:145089806-145090327:0:0-521 (2x), negative_chr3:42641896-42642557:0:0-661 (2x)
2025-03-18 19:56:29,440 WARNING: [loadMultiTrainingE

Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 0 | Number of peaks in these sequences: {}
Skipped 0/40 experiments: []
Number of sequences: 215834
Number of sequences with hits: 121702 | ratio: 0.56
Number of matches: 284647 | ratio: 1.32

Peak source: bed.tsv
Number of sequences with peaks: 215808
Number of sequences with hits on peaks: 20487 | ratio: 0.09
Number of peaks: 941
Number of hits on peaks: 20494 | ratio: 21.78

Peak source: fimo.tsv
Number of sequences with peaks: 84777
Number of sequences with hits on peaks: 28794 | ratio: 0.34
Number of peaks: 488
Number of hits on peaks: 28794 | ratio: 59.00

Peak source: mast.tsv
Number of sequences with peaks: 81851
Number of sequences with hits on peaks: 29450 | ratio: 0.36
Number of peaks: 735
Number of hits on peaks: 31378 | ratio: 42.69

Number of negative sequences: 215834
Number of negative sequences with hit

In [7]:
evaluate(experiment_dirs, 'STREME/streme_evaluator_dummymodel_test.json', 'STREME/streme_evaluator_dummymodel_negative_test.json')

2025-03-18 19:57:16,408 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)
2025-03-18 19:57:17,048 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)


[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negati

2025-03-18 19:57:23,672 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:72510161-72511226:0:0-1,065 (2x), chr6:27655855-27656374:0:0-519 (2x)
2025-03-18 19:57:23,864 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr17:72510161-72511226:0:0-1,065 (2x), negative_chr6:27655855-27656374:0:0-519 (2x)
2025-03-18 19:57:24,683 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr22:20748207-20748705:0:0-498 (2x), chr19:36208309-36208837:0:0-528 (2x), chr22:21356041-21356705:0:0-664 (2x), chr6:27100747-27101203:0:0-456 (2x)
2025-03-18 19:57:24,835 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr22:20748207-20748705:0:0-498 (2x), negative_chr19:36208309-36208837:0:0-528 (2x), negative_chr22:21356041-21356705:0:0-664 (2x), negative_chr6:27100747-27101203:0:0-456 (2x)
2025-03-18 19:57:25,938 WARNING: [loadMultiTrainin

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-18 19:57:31,769 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr6:11537625-11538146:0:0-521 (2x), chr2:175200793-175202077:0:0-1,284 (2x), chrX:37544927-37545500:0:0-573 (2x), chr1:36786772-36787466:0:0-694 (2x), chr1:16173852-16174501:0:0-649 (2x), chr1:234735500-234736178:0:0-678 (2x), chr3:38388039-38388628:0:0-589 (2x), chr19:11071274-11071738:0:0-464 (2x), chr12:124086253-124086801:0:0-548 (2x), chr3:193788616-193789324:0:0-708 (2x), chr6:144536966-144537517:0:0-551 (2x), chr6:159065362-159065766:0:0-404 (2x), chr1:112281896-112282425:0:0-529 (2x), chr7:129074035-129074584:0:0-549 (2x), chr8:98787854-98788312:0:0-458 (2x), chr5:139027661-139028005:0:0-344 (2x), chr1:33116408-33117190:0:0-782 (2x), chr12:108908671-108909083:0:0-412 (2x), chr4:90032070-90032874:0:0-804 (2x), chr7:151328925-151329711:0:0-786 (2x), chr15:41952213-41953334:0:0-1,121 (2x), chr14:100659117-100659576:0:0-459 (2x), chr15:41055323-41056071:0:0-748 (2x), chr1

...

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562CebpbIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-18 19:57:46,575 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr9:132597612-132598067:0:0-455 (2x), chr8:142427409-142428437:0:0-1,028 (2x), chr19:19516712-19517481:0:0-769 (2x), chr12:42631191-42631813:0:0-622 (2x), chr7:65447017-65447496:0:0-479 (2x), chr11:72853093-72853622:0:0-529 (2x), chr7:154793980-154794915:0:0-935 (2x), chr5:96270680-96271129:0:0-449 (2x), chr1:156662686-156663301:0:0-615 (2x), chr20:62526719-62527111:0:0-392 (2x), chr3:13036359-13036852:0:0-493 (2x), chr7:43798050-43798484:0:0-434 (2x), chr20:49307836-49308441:0:0-605 (2x), chr16:83986480-83986895:0:0-415 (2x), chr4:6784832-6785381:0:0-549 (2x), chr19:10713038-10713640:0:0-602 (2x), chr22:20849578-20850430:0:0-852 (2x), chr2:106014802-106015752:0:0-950 (2x), chr3:23847521-23848789:0:0-1,268 (2x), chr9:131418639-131419158:0:0-519 (2x), chr19:4867014-4867811:0:0-797 (2x), chr1:17764060-17764595:0:0-535 (2x), chr22:19701615-19702717:0:0-1,102 (2x), chr16:85415487

[WARNING] >>> skipping /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak as /home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsUwK562CtcfUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist


2025-03-18 19:57:55,687 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr4:128702821-128703160:0:0-339 (2x), chr7:100136668-100137233:0:0-565 (2x), chr22:19705322-19706321:0:0-999 (2x), chr1:197871671-197872361:0:0-690 (2x), chr1:43123582-43124140:0:0-558 (2x), chr17:61926089-61927170:0:0-1,081 (2x), chr2:106015376-106015966:0:0-590 (2x), chr22:20067214-20068089:0:0-875 (2x), chr9:37485657-37486115:0:0-458 (2x), chr1:155022843-155023312:0:0-469 (2x), chr11:118868258-118868750:0:0-492 (2x), chr7:135346967-135347443:0:0-476 (2x), chr11:67764043-67764415:0:0-372 (2x), chr7:148725666-148726250:0:0-584 (2x), chr19:12917200-12917697:0:0-497 (2x), chr16:89939464-89940035:0:0-571 (2x), chr13:115079619-115080065:0:0-446 (2x), chr16:81129904-81130331:0:0-427 (2x), chr11:125495140-125495604:0:0-464 (2x), chr1:12123170-12123898:0:0-728 (2x), chr19:59025184-59025713:0:0-529 (2x), chr2:232328405-232328765:0:0-360 (2x), chr5:43120774-43121302:0:0-528 (2x), chr

Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 83963 | Number of peaks in these sequences: {'bed.tsv': 84108, 'fimo.tsv': 51512, 'mast.tsv': 55107}
Skipped 7/40 experiments: ['/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak', '/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsBroadK56

---

Distribution of test sequence number and peaks per experiment, seems to be quite unevenly distributed.

In [9]:
n_test_seqs = []
n_peaks = {}
i_skipped = set()

evaluator_path = "STREME/streme_evaluator_dummymodel_test.json"
neg_evaluator_path = "STREME/streme_evaluator_dummymodel_negative_test.json"

for i, ed in enumerate(experiment_dirs):
    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())

    n_test_seqs.append( sum([len(g) for g in testdata]) )

    for g in testdata:
        for s in g:
            assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
            for e in s.genomic_elements:
                peaksrc = e.source
                if peaksrc not in n_peaks:
                    n_peaks[peaksrc] = [0]*i # in the (unlikely) case that experiments [0, i) did not have this type
                assert len(n_peaks[peaksrc]) >= i, f"{peaksrc}, {i}, {n_peaks}" # should not fail
                if len(n_peaks[peaksrc]) == i:
                    n_peaks[peaksrc].append(0)

                n_peaks[peaksrc][i] += 1

    if not (ed / evaluator_path).exists():
        i_skipped.add(i)
        continue

    if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
        i_skipped.add(i)
        continue

In [37]:
import importlib
importlib.reload(plotting)

<module 'modules.plotting' from '/home/matthis/PhD/genomegraph/learn_specific_profiles/modules/plotting.py'>

In [22]:
fig = plotting.ownPlotlyHist(lists={'all experiments': n_test_seqs, 'skipped experiments': [n_test_seqs[i] for i in i_skipped]}, binSize=500)
fig.update_layout(title_text="Experiments and their number of test sequences", xaxis_title="Number of test sequences", yaxis_title="No. exp. with this no. test sequences")
fig.show()

In [27]:
for peaksrc in n_peaks:
    fig = plotting.ownPlotlyHist(lists={'all experiments': n_peaks[peaksrc], 'skipped experiments': [n_peaks[peaksrc][i] for i in i_skipped]}, binSize=200)
    fig.update_layout(title_text=f"Experiments and their number of peaks in test sequences from {peaksrc}", xaxis_title="Number of peaks", yaxis_title="No. exp. with this no. peaks")
    fig.show()
# fig = plotting.ownPlotlyHist(lists=n_peaks, binSize=200)
# fig.update_layout(title_text="Experiments and their number of peaks in test sequences", xaxis_title="Number of peaks", yaxis_title="No. exp. with this no. peaks")
# fig.show()

---

Distribution of average peak content per experiment (percent of sequences with peaks, average number of peaks per sequence)

In [43]:
peaksrcs = set()
for ed in experiment_dirs:
    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    for g in testdata:
        for s in g:
            assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
            for e in s.genomic_elements:
                peaksrcs.add(e.source)


data = {peaksrc: {'n_test_seqs': [], 'n_test_seqs_with_peaks': [], 'n_peaks': {}} for peaksrc in peaksrcs}
for peaksrc in peaksrcs:
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))

        data[peaksrc]['n_test_seqs'].append( sum([len(g) for g in testdata]) )

        n_seqs_with_peaks = 0
        n_peaks = 0
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                hasPeak = False
                for e in s.genomic_elements:
                    if e.source == peaksrc:
                        hasPeak = True
                        n_peaks += 1
                if hasPeak:
                    n_seqs_with_peaks += 1

        data[peaksrc]['n_test_seqs_with_peaks'].append(n_seqs_with_peaks)
        data[peaksrc]['n_peaks'][ed] = n_peaks

In [44]:
for peaksrc in peaksrcs:
    data[peaksrc]['perc_seq_with_peaks'] = [100*n/nseqs for n, nseqs in zip(data[peaksrc]['n_test_seqs_with_peaks'], data[peaksrc]['n_test_seqs'])]
    data[peaksrc]['avg_num_peaks'] = [n/nseqs for n, nseqs in zip(data[peaksrc]['n_peaks'].values(), data[peaksrc]['n_test_seqs'])]

In [45]:
fig = plotting.ownPlotlyHist(lists={peaksrc: data[peaksrc]['perc_seq_with_peaks'] for peaksrc in peaksrcs}, binSize=10)
fig.update_layout(title_text="Experiments and their percentage of sequences with peaks", xaxis_title="Percentage of sequences with peaks", yaxis_title="No. exp. with this percentage")
fig.show()

In [48]:
fig = plotting.ownPlotlyHist(lists={peaksrc: data[peaksrc]['avg_num_peaks'] for peaksrc in peaksrcs}, binSize=0.1)
fig.update_layout(title_text="Experiments and their average number of peaks per sequence", xaxis_title="Average number of peaks per sequence", yaxis_title="No. exp. with this average")
fig.show()

Peak count per sequence (not per experiment)

In [35]:
data = {peaksrc: [] for peaksrc in peaksrcs}
for peaksrc in peaksrcs:
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))

        for g in testdata:
            for s in g:
                n_peaks = 0
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    if e.source == peaksrc:
                        n_peaks += 1

                data[peaksrc].append(n_peaks)

In [41]:
fig = plotting.ownPlotlyHist(lists=data, binSize=1, rel=True, xlim=(0, 6))
fig.update_layout(title_text="Number of peaks in test sequences", xaxis_title="Number of peaks", yaxis_title="No. seq. with this no. peaks")
fig.show()